In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Flatten, BatchNormalization, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import MobileNetV2

def build_transfer_learning_model(input_shape, num_classes):
    
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    
    base_model.trainable = False
    
    inputs = Input(shape=input_shape)
    
    x = base_model(inputs, training=False)
    
    x = GlobalAveragePooling2D()(x)
    
    x = Flatten()(x)
    
    x = Dense(512, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    print("\n" + "="*60)
    print("TRANSFER LEARNING MODEL - PHASE 1 READY")
    print("="*60)
    print(f"-> Base architecture: MobileNetV2 (trained on ImageNet)")
    print(f"-> Input shape required: {input_shape}")
    print(f"-> Output classes: {num_classes}")
    print(f"-> Base model frozen: YES (only custom head will train)")
    
    trainable_params = sum(tf.size(w).numpy() for w in model.trainable_weights)
    total_params = sum(tf.size(w).numpy() for w in model.weights)
    print(f"-> Trainable parameters (Phase 1): {trainable_params:,}")
    print(f"-> Total parameters: {total_params:,}")
    
    return model, base_model

def unfreeze_for_finetuning(model, base_model):
    
    base_model.trainable = True
    
    total_layers = len(base_model.layers)
    fine_tune_from = int(total_layers * 0.7)
    
    print(f"\nBase model has {total_layers} layers total")
    print(f"Freezing layers 0 to {fine_tune_from-1} (first 70%)")
    print(f"Unfreezing layers {fine_tune_from} to {total_layers-1} (last 30%)")
    
    for layer in base_model.layers[:fine_tune_from]:
        layer.trainable = False
    
    for layer in base_model.layers[fine_tune_from:]:
        layer.trainable = True
    
    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    trainable_count = sum(1 for layer in base_model.layers if layer.trainable)
    frozen_count = total_layers - trainable_count
    
    print("\n" + "="*60)
    print("TRANSFER LEARNING MODEL - PHASE 2 READY")
    print("="*60)
    print(f"-> Frozen layers: {frozen_count}")
    print(f"-> Trainable layers: {trainable_count}")
    print(f"-> Learning rate reduced to: 0.0001")
    
    return model
